# Local weather controls

Upwind fire activity surges in regional droughts and El Niño years, and drought can affect conflict directly through income. Year-month fixed effects absorb nationwide shocks, and province x calendar-month fixed effects absorb each province's typical season, but neither absorbs an unusually dry or hot month in a particular district.

This notebook adds district-month **total precipitation** and **2 m temperature** from ERA5 as controls in both stages of the IV, and re-estimates the main specifications. It reads the panel written by `wind-IV.ipynb` (`iv_panel_district.csv`), so that notebook does not need to be rerun.

## 1. Configuration and ERA5 download

The download uses the CDS API credentials in `~/.cdsapirc` and is skipped if the file already exists.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import pyfixest as pf
from scipy.spatial import cKDTree

warnings.filterwarnings('ignore')

ROOT         = '/Users/hariaksha/Documents/GitHub/climate-conflict'
WEATHER_NC   = f'{ROOT}/data/climate/weather/era5_weather_indonesia_2015_2025.nc'
WEATHER_ZIP  = f'{ROOT}/data/climate/weather/era5_weather_indonesia_2015_2025.zip'
DISTRICT_SHP = f'{ROOT}/data/administrative/gadm41_IDN_shp/gadm41_IDN_2.shp'
ACLED_XLSX   = f'{ROOT}/data/unrest/ACLED Data_2026-06-10.xlsx'
PANEL_CSV    = f'{ROOT}/analysis/iv_panel_district.csv'
OUT_CSV      = f'{ROOT}/analysis/output/weather_controls_results.csv'

START_YEAR, END_YEAR, END_MONTH = 2015, 2025, 6

if not os.path.exists(WEATHER_NC):
    # The CDS API returns precipitation and temperature as two NetCDF files in a zip
    # archive (one per ERA5 step type), so download the zip, then merge the two files.
    if not os.path.exists(WEATHER_ZIP):
        import cdsapi
        c = cdsapi.Client()  # reads ~/.cdsapirc
        c.retrieve(
            'reanalysis-era5-single-levels-monthly-means',
            {
                'product_type': 'monthly_averaged_reanalysis',
                'variable': ['total_precipitation', '2m_temperature'],
                'year': [str(y) for y in range(2015, 2026)],
                'month': [f'{m:02d}' for m in range(1, 13)],
                'time': '00:00',
                'area': [6, 95, -11, 141],
                'format': 'netcdf',
                'grid': '0.25/0.25',
            },
            WEATHER_ZIP,
        )
        print(f'Downloaded ERA5 weather data to {WEATHER_ZIP}')
    import zipfile, tempfile
    with zipfile.ZipFile(WEATHER_ZIP) as z, tempfile.TemporaryDirectory() as tmp:
        z.extractall(tmp)
        parts = [xr.open_dataset(os.path.join(tmp, n)).load() for n in z.namelist()]
    keep = lambda d: d[[v for v in ['tp', 't2m'] if v in d.data_vars]]
    xr.merge([keep(d) for d in parts], compat='override').to_netcdf(WEATHER_NC)
    print(f'Merged {len(parts)} files into {WEATHER_NC}')
else:
    print(f'ERA5 weather file already exists at {WEATHER_NC} -- skipping download.')

Merged 2 files into /Users/hariaksha/Documents/GitHub/climate-conflict/data/climate/weather/era5_weather_indonesia_2015_2025.nc


## 2. Aggregate weather to district-months

Grid cells are assigned to districts exactly as the wind fields are in `wind-IV.ipynb`: each 0.25° cell goes to the district polygon that contains it, and districts too small to contain a cell use the cell nearest their representative point.

ERA5 monthly-mean precipitation is in metres per day, so it is converted to millimetres per month. Temperature is converted from kelvin to °C.

In [2]:
districts_gdf = gpd.read_file(DISTRICT_SHP)[['NAME_1', 'NAME_2', 'geometry']].rename(
    columns={'NAME_1': 'province', 'NAME_2': 'district'})
dup = districts_gdf['district'].duplicated(keep=False)
districts_gdf.loc[dup, 'district'] = (districts_gdf.loc[dup, 'district'] + ' ('
                                      + districts_gdf.loc[dup, 'province'] + ')')
rep = districts_gdf.geometry.representative_point()
districts_gdf['dist_lat'], districts_gdf['dist_lon'] = rep.y, rep.x

ds = xr.open_dataset(WEATHER_NC)
if 'valid_time' in ds.coords:
    ds = ds.rename({'valid_time': 'time'})
wx = ds[['tp', 't2m']].to_dataframe().reset_index()
wx['time'] = pd.to_datetime(wx['time'])
wx['year'], wx['month'] = wx['time'].dt.year, wx['time'].dt.month
wx = wx[(wx['year'] >= START_YEAR) & ~((wx['year'] == END_YEAR) & (wx['month'] > END_MONTH))]
wx['precip_mm'] = wx['tp'] * 1000 * wx['time'].dt.days_in_month
wx['temp_c'] = wx['t2m'] - 273.15

grid = wx[['latitude', 'longitude']].drop_duplicates().reset_index(drop=True)
gg = gpd.GeoDataFrame(grid, geometry=gpd.points_from_xy(grid['longitude'], grid['latitude']), crs='EPSG:4326')
joined = gpd.sjoin(gg, districts_gdf[['district', 'geometry']], how='left', predicate='within')
joined = joined[~joined.index.duplicated(keep='first')]
grid['district'] = joined['district'].values
missing = districts_gdf[~districts_gdf['district'].isin(set(grid['district'].dropna()))]
_, nn = cKDTree(grid[['latitude', 'longitude']].values).query(missing[['dist_lat', 'dist_lon']].values)
fallback = grid.iloc[nn][['latitude', 'longitude']].reset_index(drop=True)
fallback['district'] = missing['district'].values
grid = pd.concat([grid.dropna(subset=['district']), fallback], ignore_index=True)
print(f'Districts with weather: {grid["district"].nunique()} ({len(missing)} via nearest-cell fallback)')

weather = (wx.merge(grid, on=['latitude', 'longitude'])
             .groupby(['district', 'year', 'month'])[['precip_mm', 'temp_c']].mean().reset_index())
print(weather.describe().round(2))

Districts with weather: 502 (94 via nearest-cell fallback)


           year     month  precip_mm    temp_c
count  63252.00  63252.00   63252.00  63252.00
mean    2019.76      6.36     253.82     25.68
std        3.04      3.45     155.80      2.02
min     2015.00      1.00       0.00     12.05
25%     2017.00      3.00     144.69     24.76
50%     2020.00      6.00     247.75     26.15
75%     2022.00      9.00     343.47     27.06
max     2025.00     12.00    1868.96     30.92


## 3. Merge into the IV panel

Controls enter as precipitation (in hundreds of mm, for readable coefficients) and temperature, each with a one-month lag, matching the lag structure of the fire variables.

In [3]:
panel = pd.read_csv(PANEL_CSV)
weather = weather.sort_values(['district', 'year', 'month'])
for v in ['precip_mm', 'temp_c']:
    weather[f'{v}_l1'] = weather.groupby('district')[v].shift(1)
panel = panel.merge(weather, on=['district', 'year', 'month'], how='left')
panel['precip'] = panel['precip_mm'] / 100
panel['precip_l1'] = panel['precip_mm_l1'] / 100
print(f'Panel: {len(panel):,} obs, {panel["district"].nunique()} districts; '
      f'missing weather: {panel[["precip", "temp_c", "precip_l1", "temp_c_l1"]].isna().any(axis=1).sum()}')
print(f'Corr(log upwind FRP, precipitation): {panel["log_upwind_frp"].corr(panel["precip"]):.3f}')
print(f'Corr(log upwind FRP, temperature):   {panel["log_upwind_frp"].corr(panel["temp_c"]):.3f}')

Panel: 60,125 obs, 481 districts; missing weather: 0
Corr(log upwind FRP, precipitation): -0.363
Corr(log upwind FRP, temperature):   0.084


## 4. Event-type outcomes

Riots/VAC and violence against civilians, from ACLED events assigned to districts by their coordinates (as in `wind-IV.ipynb`, Section 3).

In [4]:
acled = pd.read_excel(ACLED_XLSX, usecols=['event_date', 'event_type', 'latitude', 'longitude'])
acled['event_date'] = pd.to_datetime(acled['event_date'])
acled['year'], acled['month'] = acled['event_date'].dt.year, acled['event_date'].dt.month
ag = gpd.GeoDataFrame(acled, geometry=gpd.points_from_xy(acled['longitude'], acled['latitude']), crs='EPSG:4326')
j = gpd.sjoin(ag, districts_gdf[['district', 'geometry']], how='left', predicate='within')
j = j[~j.index.duplicated(keep='first')]
acled['district'] = j['district'].reindex(acled.index).values
off = acled['district'].isna()
near = gpd.sjoin_nearest(ag.loc[off].to_crs(3395), districts_gdf[['district', 'geometry']].to_crs(3395),
                         how='left', max_distance=20_000)
near = near[~near.index.duplicated(keep='first')]
acled.loc[off, 'district'] = near['district'].reindex(acled.index[off]).values

acled['riots_vac'] = acled['event_type'].isin(['Riots', 'Violence against civilians']).astype(int)
acled['vac'] = (acled['event_type'] == 'Violence against civilians').astype(int)
counts = acled.groupby(['district', 'year', 'month'])[['riots_vac', 'vac']].sum().reset_index()
panel = panel.merge(counts, on=['district', 'year', 'month'], how='left').fillna({'riots_vac': 0, 'vac': 0})
print(panel[['events', 'pv_events', 'riots_vac', 'vac']].sum())

events       21653.0
pv_events     3520.0
riots_vac     2824.0
vac           1067.0
dtype: float64


## 5. Estimates with and without weather controls

For each sample and outcome, the baseline distributed-lag IV (Table 2/3 of the paper) is re-estimated with the four weather controls added to both stages. The effective first-stage F (Montiel Olea and Pflueger, 2013) and the reduced form are reported for each specification. The last block also adds province x calendar-month fixed effects.

In [5]:
def effective_f(data, instruments, controls=(), fe='district + year_month'):
    data = data.reset_index(drop=True)
    ctrl = ' + '.join(controls)
    rhs = ' + '.join(list(instruments) + list(controls))
    fs = pf.feols(f'log_frp ~ {rhs} | {fe}', data=data, vcov={'CRV1': 'province'})
    Z = np.column_stack([pf.feols(f'{z} ~ {ctrl if ctrl else "1"} | {fe}', data=data).resid()
                         for z in instruments])
    names = list(fs.coef().index)
    idx = [names.index(z) for z in instruments]
    pi = fs.coef()[list(instruments)].to_numpy()
    V = fs._vcov[np.ix_(idx, idx)]
    S = Z.T @ Z
    return float(pi @ S @ pi / np.trace(V @ S))

INSTR = ['log_upwind_frp', 'log_upwind_frp_l1']
WEATHER = ['precip', 'precip_l1', 'temp_c', 'temp_c_l1']
OUTCOMES = ['events', 'pv_events', 'riots_vac', 'vac']

panel = panel.dropna(subset=WEATHER).reset_index(drop=True)
panel['province_month'] = panel['province'].astype(str) + '_' + panel['month'].astype(str)
n_months = panel['year_month'].nunique()
active = panel.groupby('district')['events'].apply(lambda s: (s > 0).sum())
active_30 = active[active >= 0.30 * n_months].index
samples = {'full panel': panel, 'active >= 30%': panel[panel['district'].isin(active_30)]}
print(f'Active >= 30% districts: {len(active_30)}')

specs = {
    'baseline':                    ([], 'district + year_month'),
    'weather controls':            (WEATHER, 'district + year_month'),
    'weather + province x cal. month FE': (WEATHER, 'district + year_month + province_month'),
}
rows = []
for spec, (wctrl, fe) in specs.items():
    for sample, d in samples.items():
        if spec.startswith('weather +') and sample == 'full panel':
            continue
        ctrls = ['log_frp_l1'] + wctrl
        effF = effective_f(d, INSTR, ctrls, fe)
        print(f'\n{spec} | {sample} | effective F = {effF:.1f}')
        for y in OUTCOMES:
            iv = pf.feols(f'{y} ~ {" + ".join(ctrls)} | {fe} | log_frp ~ {" + ".join(INSTR)}',
                          data=d, vcov={'CRV1': 'province'}).tidy().loc['log_frp']
            rf_rhs = ' + '.join(INSTR + wctrl)
            rf = pf.feols(f'{y} ~ {rf_rhs} | {fe}', data=d, vcov={'CRV1': 'province'}).tidy().loc['log_upwind_frp']
            print(f'  {y:10s} IV {iv["Estimate"]:+.4f} (se {iv["Std. Error"]:.4f}, p = {iv["Pr(>|t|)"]:.3f})'
                  f' | RF {rf["Estimate"]:+.4f} (se {rf["Std. Error"]:.4f}, p = {rf["Pr(>|t|)"]:.3f})')
            rows.append({'spec': spec, 'sample': sample, 'outcome': y, 'effective_F': effF, 'N': len(d),
                         'iv_coef': iv['Estimate'], 'iv_se': iv['Std. Error'], 'iv_p': iv['Pr(>|t|)'],
                         'rf_coef': rf['Estimate'], 'rf_se': rf['Std. Error'], 'rf_p': rf['Pr(>|t|)']})

results = pd.DataFrame(rows)
results.to_csv(OUT_CSV, index=False)
print(f'\nSaved: {OUT_CSV}')

Active >= 30% districts: 85



baseline | full panel | effective F = 75.7


  events     IV -0.0069 (se 0.0084, p = 0.422) | RF -0.0024 (se 0.0020, p = 0.237)


  pv_events  IV -0.0022 (se 0.0023, p = 0.337) | RF -0.0008 (se 0.0006, p = 0.210)


  riots_vac  IV -0.0025 (se 0.0024, p = 0.306) | RF -0.0008 (se 0.0006, p = 0.186)


  vac        IV -0.0026 (se 0.0018, p = 0.147) | RF -0.0007 (se 0.0005, p = 0.150)

baseline | active >= 30% | effective F = 26.4
  events     IV -0.1870 (se 0.0468, p = 0.000) | RF -0.0261 (se 0.0078, p = 0.002)
  pv_events  IV -0.0469 (se 0.0221, p = 0.042) | RF -0.0072 (se 0.0039, p = 0.078)
  riots_vac  IV -0.0514 (se 0.0203, p = 0.017) | RF -0.0079 (se 0.0037, p = 0.039)


... (4 outputs left out)

  vac        IV -0.0031 (se 0.0019, p = 0.099) | RF -0.0006 (se 0.0003, p = 0.113)

weather controls | active >= 30% | effective F = 15.6
  events     IV -0.2579 (se 0.0669, p = 0.001) | RF -0.0262 (se 0.0077, p = 0.002)
  pv_events  IV -0.0725 (se 0.0276, p = 0.013) | RF -0.0078 (se 0.0041, p = 0.063)


  riots_vac  IV -0.0747 (se 0.0259, p = 0.007) | RF -0.0081 (se 0.0038, p = 0.041)
  vac        IV -0.0479 (se 0.0169, p = 0.008) | RF -0.0051 (se 0.0025, p = 0.051)



weather + province x cal. month FE | active >= 30% | effective F = 7.1
  events     IV -0.3005 (se 0.1039, p = 0.007) | RF -0.0262 (se 0.0100, p = 0.013)
  pv_events  IV -0.1244 (se 0.0298, p = 0.000) | RF -0.0116 (se 0.0053, p = 0.035)


  riots_vac  IV -0.1089 (se 0.0272, p = 0.000) | RF -0.0106 (se 0.0045, p = 0.026)
  vac        IV -0.0710 (se 0.0128, p = 0.000) | RF -0.0068 (se 0.0029, p = 0.024)

Saved: /Users/hariaksha/Documents/GitHub/climate-conflict/analysis/output/weather_controls_results.csv


## 6. Do the weather variables themselves predict fire and conflict?

Coefficients on the weather controls in the first stage and in the all-events reduced form, conflict-active sample.

In [6]:
d = samples['active >= 30%']
fs = pf.feols(f'log_frp ~ {" + ".join(INSTR + ["log_frp_l1"] + WEATHER)} | district + year_month',
              data=d, vcov={'CRV1': 'province'})
rf = pf.feols(f'events ~ {" + ".join(INSTR + WEATHER)} | district + year_month',
              data=d, vcov={'CRV1': 'province'})
print('First stage (log FRP):')
print(fs.tidy().loc[WEATHER, ['Estimate', 'Std. Error', 'Pr(>|t|)']].round(4))
print('\nReduced form (events):')
print(rf.tidy().loc[WEATHER, ['Estimate', 'Std. Error', 'Pr(>|t|)']].round(4))

First stage (log FRP):
             Estimate  Std. Error  Pr(>|t|)
Coefficient                                
precip        -0.1380      0.0221    0.0000
precip_l1     -0.0662      0.0209    0.0033
temp_c         0.2909      0.0585    0.0000
temp_c_l1     -0.3218      0.0448    0.0000

Reduced form (events):
             Estimate  Std. Error  Pr(>|t|)
Coefficient                                
precip        -0.0102      0.0197    0.6098
precip_l1     -0.0180      0.0277    0.5198
temp_c        -0.0909      0.0415    0.0358
temp_c_l1      0.1717      0.0685    0.0175
